# Chapter 6, Guide 3: Amplitude amplification, with derandomization

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 6,
"Building amplitude amplification" and "Derandomizing, and testing what you built".

What the book says you must supply: $\mathcal{A}$ (unitary state preparation, not a
measurement), $\mathcal{A}^\dagger$ (reverse the gate order and dagger each gate; Qiskit's
`inverse()` does exactly that), and $\mathcal{S}_g$ (mark the good states). The
reflection $\mathcal{S}_0$ is the same multi-controlled-$Z$ sandwich as Grover's diffuser.
Then $\mathcal{Q} = \mathcal{A}\mathcal{S}_0\mathcal{A}^\dagger\mathcal{S}_g$.

Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

bq = lambda j, tot: tot - 1 - j

## A non-uniform state preparation

$\mathcal{A} = H \otimes H \otimes R_y(\phi)$ on three wires, and the good set is the
single string $|111\rangle$, so $g = \tfrac14\sin^2(\phi/2)$. Choosing $\phi$ sets $g$
anywhere in $(0, 1/4]$, which is enough to see the rotation and the overshoot.

In [2]:
n = 3
phi = 1.2
g_exact = 0.25 * np.sin(phi / 2) ** 2
print("g =", g_exact)


def A_prep(tot, wires):
    qc = QuantumCircuit(tot, name="A")
    qc.h(wires[0]); qc.h(wires[1]); qc.ry(phi, wires[2])
    return qc


def S_good(tot, wires, anc, extra_ctrl=None):
    """Mark |111> (times |1> on extra_ctrl if given) by a bit flip onto the |-> ancilla."""
    qc = QuantumCircuit(tot, name="S_g")
    ctrls = list(wires) + ([] if extra_ctrl is None else [extra_ctrl])
    qc.mcx(ctrls, anc)
    return qc


def S_zero(tot, wires):
    """-(2|0><0| - I) on the given wires: X^n C^{n-1}(Z) X^n (global sign irrelevant here)."""
    qc = QuantumCircuit(tot, name="S_0")
    qc.x(wires)
    t = wires[-1]
    qc.h(t); qc.mcx(wires[:-1], t); qc.h(t)
    qc.x(wires)
    return qc


def Q_circuit(tot, wires, anc, extra_ctrl=None, g_target=None):
    """One amplitude-amplification iteration Q = A S_0 A^dagger S_g."""
    qc = QuantumCircuit(tot)
    allw = list(wires) + ([] if extra_ctrl is None else [extra_ctrl])
    qc.compose(S_good(tot, wires, anc, extra_ctrl), inplace=True)
    qc.compose(A_full(tot, wires, extra_ctrl, g_target).inverse(), inplace=True)
    qc.compose(S_zero(tot, allw), inplace=True)
    qc.compose(A_full(tot, wires, extra_ctrl, g_target), inplace=True)
    return qc


def A_full(tot, wires, extra_ctrl=None, g_target=None):
    """A, or A tensor the derandomizing rotation R on extra_ctrl."""
    qc = A_prep(tot, wires)
    if extra_ctrl is not None:
        qc.ry(2 * np.arcsin(np.sqrt(g_target / g_exact)), extra_ctrl)
    return qc

g = 0.07970528069041581


## Test: $\Pr(\text{good})$ against $r$ matches $\sin^2((2r+1)\theta)$, $\theta=\arcsin\sqrt g$

In [3]:
tot = n + 1
wires = [bq(j, tot) for j in range(n)]
anc = bq(n, tot)


def run_aa(r):
    qc = QuantumCircuit(tot)
    qc.x(anc); qc.h(anc)                       # |->
    qc.compose(A_full(tot, wires), inplace=True)
    for _ in range(r):
        qc.compose(Q_circuit(tot, wires, anc), inplace=True)
    sv = Statevector(qc)
    probs = sv.probabilities(wires[::-1])     # indexed by the book integer of the 3 wires
    return probs[0b111]


theta = np.arcsin(np.sqrt(g_exact))
rs = np.arange(0, 8)
sim = np.array([run_aa(r) for r in rs])
th = np.sin((2 * rs + 1) * theta) ** 2
for r, a, b in zip(rs, sim, th):
    print(f"r={r}: {a:.5f}  vs  {b:.5f}")
assert np.allclose(sim, th, atol=1e-10)
print("Test passed")
plt.plot(rs, sim, "o", label="circuit"); plt.plot(rs, th, "-", label=r"$\sin^2((2r+1)\theta)$")
plt.xlabel("r"); plt.ylabel("Pr(good)"); plt.legend(); plt.show()

r=0: 0.07971  vs  0.07971
r=1: 0.57298  vs  0.57298
r=2: 0.98060  vs  0.98060
r=3: 0.82417  vs  0.82417
r=4: 0.28728  vs  0.28728
r=5: 0.00005  vs  0.00005
r=6: 0.29957  vs  0.29957
r=7: 0.83433  vs  0.83433
Test passed


## Derandomization

$r = \lceil \pi/(4\arcsin\sqrt{g_0}) - 1/2 \rceil$, then $g_0' = \sin^2\!\big(\pi/(2(2r+1))\big)$,
then $R = R_y(2\arcsin\sqrt{g_0'/g_0})$ on a fresh ancilla, and the oracle gains one
control line requiring that ancilla to be $|1\rangle$. The book's test: the probability
at the chosen $r$ must be $1$ to numerical precision.

In [4]:
g0 = g_exact
r_d = int(np.ceil(np.pi / (4 * np.arcsin(np.sqrt(g0))) - 0.5))
g0p = np.sin(np.pi / (2 * (2 * r_d + 1))) ** 2
print(f"r = {r_d}, g0 = {g0:.5f}, g0' = {g0p:.5f}  (g0' <= g0: {g0p <= g0})")

tot2 = n + 2
wires2 = [bq(j, tot2) for j in range(n)]
b = bq(n, tot2)          # derandomizing ancilla
anc2 = bq(n + 1, tot2)   # oracle |-> ancilla

def run_derandomized(r):
    qc = QuantumCircuit(tot2)
    qc.x(anc2); qc.h(anc2)
    qc.compose(A_full(tot2, wires2, b, g0p), inplace=True)
    for _ in range(r):
        qc.compose(Q_circuit(tot2, wires2, anc2, extra_ctrl=b, g_target=g0p), inplace=True)
    sv = Statevector(qc)
    probs = sv.probabilities([b] + wires2[::-1])   # index = 2*(book int) + b
    return probs[2 * 0b111 + 1]


for r in range(r_d + 2):
    print(f"r={r}: Pr(good and b=1) = {run_derandomized(r):.10f}")
assert abs(run_derandomized(r_d) - 1) < 1e-10
print(f"Test passed: probability 1 at r = {r_d}")

r = 3, g0 = 0.07971, g0' = 0.04952  (g0' <= g0: True)
r=0: Pr(good and b=1) = 0.0495155660
r=1: Pr(good and b=1) = 0.3887395330
r=2: Pr(good and b=1) = 0.8117449009
r=3: Pr(good and b=1) = 1.0000000000
r=4: Pr(good and b=1) = 0.8117449009
Test passed: probability 1 at r = 3


Each $\mathcal{Q}$ contains one $\mathcal{A}$ and one $\mathcal{A}^\dagger$; the cost per
iteration is twice the state preparation plus two reflections, which is why a cheap
$\mathcal{A}$ matters more than a cheap oracle in most applications.